<a href="https://colab.research.google.com/github/LeeHJ030417/Machine-Learning-Course-Project/blob/main/%EA%B8%B0%EA%B3%84%ED%95%99%EC%8A%B5_26_05_19.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AG News Classifier — v26_05_19 (Feedback-Reflected Full Version)

## 26_05_17 피드백 반영 사항 요약

### Proposal 대비 추가 구현 항목
| # | 항목 | 위치 |
|---|------|------|
| 1 | **Random masking** (3 models) | Cell 10 |
| 2 | **Attribution-based masking** (3 models, leave-one-out) | Cell 10 |
| 3 | **Source marker removal** (e.g. `(Reuters)`, `(AP)`) | Cell 8 |
| 4 | **Named-Entity masking** (spaCy) | Cell 8 |
| 5 | **Punctuation removal** | Cell 8 |
| 6 | **Lowercasing** | Cell 8 |
| 7 | **Whitespace normalization** | Cell 8 |
| 8 | **Learning curve analysis** (subsample 학습) | Cell 11 |
| 9 | **High-confidence wrong prediction 정성 분석** | Cell 9 |
| 10 | TF-IDF + LR top coefficient feature 분석 | Cell 12 |
| 11 | TextCNN filter activation / top phrase 분석 | Cell 13 |
| 12 | DistilBERT attention 분석 | Cell 14 |
| 13 | Per-class precision / recall 출력 | Cell 7 |
| 14 | Ambiguous / mixed-signal example 분석 | Cell 15 |

## 실행 순서
1. Cell 1 (패키지 설치) — 세션당 1회
2. Cell 2~4 (셋업) — 런타임 재시작 후 매번
3. Cell 5 (학습) — 1회만, 체크포인트 자동 저장
4. Cell 6~16 (평가/분석) — 순서 무관, 자동 복구

In [ ]:
# %% [Cell 1] Install Dependencies
# !pip install -q datasets transformers accelerate scikit-learn \
#              torch matplotlib seaborn tqdm joblib spacy
# !python -m spacy download en_core_web_sm -q

In [ ]:
# %% [Cell 2] Imports · Global Config · Constants
import os, re, random, string, urllib.request, warnings, pickle, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                              recall_score, confusion_matrix,
                              classification_report)
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

import transformers
from transformers import (DistilBertTokenizerFast,
                           DistilBertForSequenceClassification,
                           get_linear_schedule_with_warmup)

import spacy

warnings.filterwarnings('ignore')
transformers.logging.set_verbosity_error()

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

# ── 전역 상수 ────────────────────────────────────────────────────────────────
CLASSES     = ['World', 'Sports', 'Business', 'Sci/Tech']
NUM_CLASSES = 4
TEXT_COL    = 'title_desc'
LABEL_COL   = 'label'
INPUT_MODES = ['title', 'description', 'title_desc']

# ── 저장 경로 ────────────────────────────────────────────────────────────────
DATA_DIR     = '/content/data'
CKPT_DIR     = '/content/checkpoints'
LC_CKPT_DIR  = '/content/checkpoints/learning_curve'
os.makedirs(DATA_DIR,    exist_ok=True)
os.makedirs(CKPT_DIR,    exist_ok=True)
os.makedirs(LC_CKPT_DIR, exist_ok=True)

DF_PATH     = os.path.join(DATA_DIR, 'dataframes.pkl')
TFIDF_PATH  = lambda mode: os.path.join(CKPT_DIR, f'tfidf_{mode}.joblib')
CNN_PATH    = lambda mode: os.path.join(CKPT_DIR, f'cnn_{mode}.pt')
CNN_TOK_PATH= lambda mode: os.path.join(CKPT_DIR, f'cnn_tok_{mode}.pkl')
BERT_PATH   = lambda mode: os.path.join(CKPT_DIR, f'bert_{mode}.pt')

print("Setup complete.")

In [ ]:
# %% [Cell 3] Load Data (auto-recovery via disk)
TRAIN_CSV = os.path.join(DATA_DIR, 'ag_train.csv')
TEST_CSV  = os.path.join(DATA_DIR, 'ag_test.csv')
TRAIN_URL = ("https://raw.githubusercontent.com/"
             "mhjabreel/CharCnn_Keras/master/data/ag_news_csv/train.csv")
TEST_URL  = ("https://raw.githubusercontent.com/"
             "mhjabreel/CharCnn_Keras/master/data/ag_news_csv/test.csv")

if not os.path.exists(TRAIN_CSV):
    print("Downloading train.csv ..."); urllib.request.urlretrieve(TRAIN_URL, TRAIN_CSV)
if not os.path.exists(TEST_CSV):
    print("Downloading test.csv  ..."); urllib.request.urlretrieve(TEST_URL,  TEST_CSV)


def _clean_text(t):
    t = re.sub(r'[^A-Za-z0-9\s(),!?]', ' ', str(t))
    t = re.sub(r'\s{2,}', ' ', t)
    return t.strip().lower()


def _load_csv(path):
    df = pd.read_csv(path, header=None, names=[LABEL_COL, 'title', 'description'])
    df[LABEL_COL]    = df[LABEL_COL] - 1
    df['title']      = df['title'].fillna('').apply(_clean_text)
    df['description']= df['description'].fillna('').apply(_clean_text)
    df['title_desc'] = (df['title'] + ' ' + df['description']).str.strip()
    # Robustness 실험을 위해 ORIGINAL(클렌징 전) 텍스트도 따로 보관
    raw_title = pd.read_csv(path, header=None,
                            names=[LABEL_COL,'title','description'])['title'].fillna('')
    raw_desc  = pd.read_csv(path, header=None,
                            names=[LABEL_COL,'title','description'])['description'].fillna('')
    df['title_desc_raw'] = (raw_title + ' ' + raw_desc).str.strip()
    return df


if os.path.exists(DF_PATH):
    print(f"Loading existing dataframes from {DF_PATH} ...")
    with open(DF_PATH, 'rb') as f:
        _b = pickle.load(f)
    train_df = _b['train']; val_df = _b['val']; test_df = _b['test']
else:
    print("Building dataframes from CSV ...")
    full = _load_csv(TRAIN_CSV)
    test_df = _load_csv(TEST_CSV)
    train_df, val_df = train_test_split(
        full, test_size=0.10, random_state=SEED, stratify=full[LABEL_COL])
    train_df = train_df.reset_index(drop=True)
    val_df   = val_df.reset_index(drop=True)
    with open(DF_PATH, 'wb') as f:
        pickle.dump({'train': train_df, 'val': val_df, 'test': test_df}, f)
    print(f"Saved → {DF_PATH}")

print(f"Train: {len(train_df):,}  |  Val: {len(val_df):,}  |  Test: {len(test_df):,}")

In [ ]:
# %% [Cell 4] Model Classes
def compute_metrics(labels, preds):
    labels = list(labels); preds = list(preds)
    return {
        'acc':       accuracy_score(labels, preds),
        'macro_f1':  f1_score(labels, preds, average='macro'),
        'per_f1':    f1_score(labels, preds, average=None),
        'per_prec':  precision_score(labels, preds, average=None, zero_division=0),
        'per_rec':   recall_score(labels, preds, average=None, zero_division=0),
        'preds':     np.array(preds),
    }

# ──────────────── (1) TF-IDF + LR ────────────────
def build_tfidf_pipeline():
    return Pipeline([
        ('tfidf', TfidfVectorizer(max_features=50_000, ngram_range=(1, 2),
                                   sublinear_tf=True, strip_accents='unicode')),
        ('clf',   LogisticRegression(C=5.0, max_iter=1000, solver='lbfgs',
                                      multi_class='multinomial',
                                      random_state=SEED, n_jobs=-1)),
    ])

def train_tfidf_lr(texts, labels):
    pipe = build_tfidf_pipeline(); pipe.fit(texts, labels); return pipe

# ──────────────── (2) TextCNN ────────────────
CNN_MAX_LEN, CNN_EMBED, CNN_NF = 200, 100, 100
CNN_FILTER_SIZES = [3, 4, 5]
CNN_DROPOUT, CNN_BS, CNN_EPOCHS, CNN_LR = 0.5, 64, 5, 1e-3


class CNNTokenizer:
    def __init__(self, texts=None, min_freq=5):
        self.word_to_idx = {'<unk>': 0, '<pad>': 1}
        if texts is not None:
            cnt = Counter(w for t in texts for w in t.split())
            for w, c in cnt.items():
                if c >= min_freq:
                    self.word_to_idx[w] = len(self.word_to_idx)
        self.idx_to_word = {i: w for w, i in self.word_to_idx.items()}

    @property
    def vocab_size(self):
        return len(self.word_to_idx)

    def encode(self, text, max_len):
        ids = [self.word_to_idx.get(w, 0) for w in text.split()[:max_len]]
        ids += [1] * (max_len - len(ids))
        return torch.tensor(ids, dtype=torch.long)


class CNNDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.data = [
            {'input_ids': tokenizer.encode(t, max_len),
             'labels':    torch.tensor(l, dtype=torch.long)}
            for t, l in zip(texts, labels)]
    def __len__(self):        return len(self.data)
    def __getitem__(self, i): return self.data[i]


class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=CNN_EMBED, n_filters=CNN_NF,
                 filter_sizes=CNN_FILTER_SIZES, n_classes=NUM_CLASSES,
                 dropout=CNN_DROPOUT, pad_idx=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.filter_sizes = filter_sizes
        self.convs = nn.ModuleList([
            nn.Conv2d(1, n_filters, (fs, embed_dim)) for fs in filter_sizes])
        self.fc = nn.Linear(len(filter_sizes) * n_filters, n_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, return_activations=False):
        emb = self.embedding(x).unsqueeze(1)
        pooled, activations = [], []
        for conv in self.convs:
            c = F.relu(conv(emb)).squeeze(3)
            p = F.max_pool1d(c, c.size(2)).squeeze(2)
            pooled.append(p)
            if return_activations:
                activations.append(c)   # (B, n_filters, L-k+1)
        out = self.fc(self.dropout(torch.cat(pooled, dim=1)))
        if return_activations:
            return out, activations
        return out


def train_cnn(texts, labels, val_texts=None, val_labels=None,
              tokenizer=None, max_len=CNN_MAX_LEN, epochs=CNN_EPOCHS, verbose=True):
    if tokenizer is None:
        tokenizer = CNNTokenizer(texts, min_freq=5)
    tr_loader = DataLoader(CNNDataset(texts, labels, tokenizer, max_len),
                            batch_size=CNN_BS, shuffle=True, num_workers=2)
    val_loader = (DataLoader(CNNDataset(val_texts, val_labels, tokenizer, max_len),
                              batch_size=CNN_BS, shuffle=False, num_workers=2)
                  if val_texts is not None else None)

    model = TextCNN(tokenizer.vocab_size).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=CNN_LR)
    criterion = nn.CrossEntropyLoss()
    best_f1, best_state = 0.0, None

    for epoch in range(1, epochs + 1):
        model.train()
        tl = ta = 0.0
        for batch in tqdm(tr_loader, desc=f'  [CNN] Ep {epoch}/{epochs}', leave=False):
            ids = batch['input_ids'].to(DEVICE); ys = batch['labels'].to(DEVICE)
            optimizer.zero_grad()
            out = model(ids); loss = criterion(out, ys)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            tl += loss.item(); ta += (out.argmax(1) == ys).float().mean().item()
        msg = f'  Ep {epoch}: loss={tl/len(tr_loader):.4f} acc={ta/len(tr_loader):.4f}'
        if val_loader is not None:
            v = eval_cnn(model, val_loader)
            msg += f'  val_acc={v["acc"]:.4f}  val_f1={v["macro_f1"]:.4f}'
            if v['macro_f1'] > best_f1:
                best_f1 = v['macro_f1']
                best_state = {k: t.clone() for k, t in model.state_dict().items()}
        if verbose:
            print(msg)
    if best_state is not None:
        model.load_state_dict(best_state)
        if verbose:
            print(f'  ✓ Best val Macro-F1: {best_f1:.4f}')
    return model, tokenizer


def eval_cnn(model, loader, get_probs=False):
    model.eval()
    preds, labels, probs = [], [], []
    with torch.no_grad():
        for batch in loader:
            ids = batch['input_ids'].to(DEVICE); ys = batch['labels'].to(DEVICE)
            out = model(ids)
            preds.extend(out.argmax(1).cpu().numpy())
            labels.extend(ys.cpu().numpy())
            if get_probs:
                probs.extend(F.softmax(out, dim=1).cpu().numpy())
    r = compute_metrics(labels, preds)
    if get_probs: r['probs'] = np.array(probs)
    return r


def predict_cnn(model, texts, tokenizer, max_len, get_probs=True):
    ds = CNNDataset(texts, [0]*len(texts), tokenizer, max_len)
    loader = DataLoader(ds, batch_size=CNN_BS, shuffle=False, num_workers=2)
    return eval_cnn(model, loader, get_probs=get_probs)

# ──────────────── (3) DistilBERT ────────────────
BERT_MODEL, BERT_MAX_LEN, BERT_BS, BERT_EPOCHS, BERT_LR = \
    'distilbert-base-uncased', 128, 16, 3, 2e-5
bert_tokenizer = DistilBertTokenizerFast.from_pretrained(BERT_MODEL)


class BertDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=BERT_MAX_LEN):
        enc = tokenizer(list(texts), max_length=max_len, truncation=True,
                        padding='max_length', return_tensors='pt')
        self.input_ids      = enc['input_ids']
        self.attention_mask = enc['attention_mask']
        self.labels         = torch.tensor(list(labels), dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        return {'input_ids': self.input_ids[i],
                'attention_mask': self.attention_mask[i],
                'labels': self.labels[i]}


def build_bert_model(output_attentions=False):
    return DistilBertForSequenceClassification.from_pretrained(
        BERT_MODEL, num_labels=NUM_CLASSES,
        output_attentions=output_attentions).to(DEVICE)


def train_bert(texts, labels, val_texts=None, val_labels=None,
               epochs=BERT_EPOCHS, verbose=True):
    model = build_bert_model()
    tr_loader = DataLoader(BertDataset(texts, labels, bert_tokenizer),
                            batch_size=BERT_BS, shuffle=True,
                            num_workers=2, pin_memory=True)
    total_steps = len(tr_loader) * epochs
    optimizer = optim.AdamW(model.parameters(), lr=BERT_LR, weight_decay=0.01)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=total_steps//10, num_training_steps=total_steps)
    best_f1, best_state = 0.0, None

    for epoch in range(1, epochs + 1):
        model.train()
        tl = 0.0
        for batch in tqdm(tr_loader, desc=f'  [BERT] Ep {epoch}/{epochs}', leave=False):
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            ys   = batch['labels'].to(DEVICE)
            out  = model(input_ids=ids, attention_mask=mask, labels=ys)
            optimizer.zero_grad()
            out.loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()
            tl += out.loss.item()
        msg = f'  Ep {epoch}: loss={tl/len(tr_loader):.4f}'
        if val_texts is not None:
            v = predict_bert(model, val_texts, val_labels, get_probs=False)
            msg += f'  val_acc={v["acc"]:.4f}  val_f1={v["macro_f1"]:.4f}'
            if v['macro_f1'] > best_f1:
                best_f1 = v['macro_f1']
                best_state = {k: t.clone() for k, t in model.state_dict().items()}
        if verbose:
            print(msg)
    if best_state is not None:
        model.load_state_dict(best_state)
        if verbose:
            print(f'  ✓ Best val Macro-F1: {best_f1:.4f}')
    return model


def predict_bert(model, texts, labels=None, batch_size=64, get_probs=True):
    if labels is None:
        labels = [0]*len(texts)
    ds = BertDataset(texts, labels, bert_tokenizer)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=2)
    model.eval()
    preds, ls, probs = [], [], []
    with torch.no_grad():
        for batch in loader:
            logits = model(input_ids=batch['input_ids'].to(DEVICE),
                            attention_mask=batch['attention_mask'].to(DEVICE)).logits
            preds.extend(logits.argmax(1).cpu().numpy())
            ls.extend(batch['labels'].numpy())
            if get_probs:
                probs.extend(F.softmax(logits, dim=1).cpu().numpy())
    r = compute_metrics(ls, preds)
    if get_probs: r['probs'] = np.array(probs)
    return r


print("✓ All model classes loaded.")

In [ ]:
# %% [Cell 5] Train All Models + Save Checkpoints
assert 'train_df' in dir(), "❌ Cell 3 먼저 실행."

trained_models = {}
for mode in INPUT_MODES:
    print(f"\n{'='*60}\n  Training mode = {mode}\n{'='*60}")
    bundle = {}
    tr_texts  = train_df[mode].tolist();  tr_labels = train_df[LABEL_COL].tolist()
    vl_texts  = val_df[mode].tolist();    vl_labels = val_df[LABEL_COL].tolist()

    # TF-IDF
    p = TFIDF_PATH(mode)
    if os.path.exists(p):
        print(f"  [TFIDF] load {p}"); bundle['tfidf'] = joblib.load(p)
    else:
        print("  [TFIDF] training ...")
        bundle['tfidf'] = train_tfidf_lr(tr_texts, tr_labels)
        joblib.dump(bundle['tfidf'], p); print(f"  saved → {p}")

    # TextCNN
    cp, tp = CNN_PATH(mode), CNN_TOK_PATH(mode)
    raw_max = max(len(t.split()) for t in tr_texts + test_df[mode].tolist())
    cnn_maxlen = min(raw_max, CNN_MAX_LEN)
    if os.path.exists(cp) and os.path.exists(tp):
        print(f"  [CNN] load {cp}")
        with open(tp, 'rb') as f: cnn_tok = pickle.load(f)
        cnn_model = TextCNN(cnn_tok.vocab_size).to(DEVICE)
        cnn_model.load_state_dict(torch.load(cp, map_location=DEVICE))
    else:
        print("  [CNN] training ...")
        cnn_model, cnn_tok = train_cnn(tr_texts, tr_labels, vl_texts, vl_labels,
                                        max_len=cnn_maxlen)
        torch.save(cnn_model.state_dict(), cp)
        with open(tp, 'wb') as f: pickle.dump(cnn_tok, f)
        print(f"  saved → {cp}")
    bundle.update({'cnn': cnn_model, 'cnn_tok': cnn_tok, 'cnn_maxlen': cnn_maxlen})

    # DistilBERT
    bp = BERT_PATH(mode)
    if os.path.exists(bp):
        print(f"  [BERT] load {bp}")
        bert = build_bert_model()
        bert.load_state_dict(torch.load(bp, map_location=DEVICE))
    else:
        print("  [BERT] training ...")
        bert = train_bert(tr_texts, tr_labels, vl_texts, vl_labels)
        torch.save(bert.state_dict(), bp); print(f"  saved → {bp}")
    bundle['bert'] = bert

    trained_models[mode] = bundle
print(f"\n✓ All models ready. Modes: {list(trained_models.keys())}")

In [ ]:
# %% [Cell 6] Auto-Recover + Evaluate on Test Set
def _ensure_data():
    global train_df, val_df, test_df
    if 'train_df' not in globals() or 'test_df' not in globals():
        print("⚠ DataFrames missing → reload from disk")
        with open(DF_PATH, 'rb') as f:
            b = pickle.load(f)
        train_df, val_df, test_df = b['train'], b['val'], b['test']

def _ensure_models():
    global trained_models
    if 'trained_models' in globals() and len(trained_models) == len(INPUT_MODES):
        return
    print("⚠ trained_models missing → reload from checkpoints")
    trained_models = {}
    for mode in INPUT_MODES:
        b = {'tfidf': joblib.load(TFIDF_PATH(mode))}
        with open(CNN_TOK_PATH(mode), 'rb') as f:
            b['cnn_tok'] = pickle.load(f)
        b['cnn'] = TextCNN(b['cnn_tok'].vocab_size).to(DEVICE)
        b['cnn'].load_state_dict(torch.load(CNN_PATH(mode), map_location=DEVICE))
        raw_max = max(len(t.split())
                       for t in train_df[mode].tolist() + test_df[mode].tolist())
        b['cnn_maxlen'] = min(raw_max, CNN_MAX_LEN)
        b['bert'] = build_bert_model()
        b['bert'].load_state_dict(torch.load(BERT_PATH(mode), map_location=DEVICE))
        trained_models[mode] = b
    print(f"  ✓ Restored: {list(trained_models.keys())}")

_ensure_data(); _ensure_models()

all_results = defaultdict(dict)
for mode in INPUT_MODES:
    b = trained_models[mode]
    te_texts, te_labels = test_df[mode].tolist(), test_df[LABEL_COL].tolist()

    probs = b['tfidf'].predict_proba(te_texts)
    r = compute_metrics(te_labels, np.argmax(probs, 1)); r['probs'] = probs
    all_results['TF-IDF+LR'][mode] = r
    print(f"[TFIDF+LR  | {mode}]  Acc={r['acc']:.4f}  Macro-F1={r['macro_f1']:.4f}")

    rr = predict_cnn(b['cnn'], te_texts, b['cnn_tok'], b['cnn_maxlen'])
    r = compute_metrics(te_labels, rr['preds']); r['probs'] = rr['probs']
    all_results['TextCNN'][mode] = r
    print(f"[TextCNN   | {mode}]  Acc={r['acc']:.4f}  Macro-F1={r['macro_f1']:.4f}")

    r = predict_bert(b['bert'], te_texts, te_labels, get_probs=True)
    all_results['DistilBERT'][mode] = r
    print(f"[DistilBERT| {mode}]  Acc={r['acc']:.4f}  Macro-F1={r['macro_f1']:.4f}")

tfidf_results, textcnn_results, distilbert_results = (
    all_results['TF-IDF+LR'], all_results['TextCNN'], all_results['DistilBERT'])
trained_probs = {m: all_results[m][TEXT_COL]['probs']
                 for m in ['TF-IDF+LR', 'TextCNN', 'DistilBERT']}

print("\n✓ Evaluation complete.")

In [ ]:
# %% [Cell 7] Experiment 1 — Input Ablation + Per-class Precision/Recall
# ──── 피드백 반영: per-class precision/recall 출력 추가 ────
if 'all_results' not in dir():
    raise RuntimeError("Cell 6 먼저 실행.")

# (a) Ablation summary
rows = []
for mode in INPUT_MODES:
    for mname in ['TF-IDF+LR', 'TextCNN', 'DistilBERT']:
        r = all_results[mname][mode]
        row = {'Model': mname, 'Input': mode,
               'Accuracy': r['acc'], 'Macro-F1': r['macro_f1']}
        for i, cls in enumerate(CLASSES):
            row[f'F1-{cls}'] = r['per_f1'][i]
        rows.append(row)
summary_df = pd.DataFrame(rows).round(4)
print("=== Input Ablation Summary ===")
print(summary_df.to_string(index=False))

# (b) Per-class precision/recall (title_desc 기준) — 피드백 반영
print(f"\n=== Per-class Precision / Recall / F1  ({TEXT_COL}) ===")
for mname in ['TF-IDF+LR', 'TextCNN', 'DistilBERT']:
    r = all_results[mname][TEXT_COL]
    print(f"\n[{mname}]")
    print(f"  {'Class':<10}  {'Prec':>7}  {'Recall':>7}  {'F1':>7}")
    for i, cls in enumerate(CLASSES):
        print(f"  {cls:<10}  {r['per_prec'][i]:>7.4f}  "
              f"{r['per_rec'][i]:>7.4f}  {r['per_f1'][i]:>7.4f}")

# (c) Ablation bar plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, metric in zip(axes, ['Accuracy', 'Macro-F1']):
    pivot = summary_df.pivot(index='Input', columns='Model', values=metric)
    pivot.plot(kind='bar', ax=ax, colormap='Set2', edgecolor='black', width=0.7)
    ax.set_title(f'{metric} by Input Mode'); ax.set_ylim(0.75, 1.02)
    ax.set_xticklabels(pivot.index, rotation=15)
    for c in ax.containers:
        ax.bar_label(c, fmt='%.3f', fontsize=7, padding=2)
plt.suptitle('Input Ablation — All Models', fontsize=13)
plt.tight_layout(); plt.show()

# (d) Confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, mname in zip(axes, ['TF-IDF+LR', 'TextCNN', 'DistilBERT']):
    r = all_results[mname][TEXT_COL]
    cm = confusion_matrix(test_df[LABEL_COL], r['preds'])
    cmn = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cmn, annot=True, fmt='.2f', ax=ax,
                xticklabels=CLASSES, yticklabels=CLASSES,
                cmap='Blues', vmin=0, vmax=1)
    ax.set_title(f'{mname} | {TEXT_COL}\nAcc={r["acc"]:.4f}')
    ax.set_ylabel('True'); ax.set_xlabel('Predicted')
plt.suptitle('Normalised Confusion Matrices', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# %% [Cell 8] Experiment 2 — Robustness (6 Perturbations)
# ──── 피드백 반영: source_marker_removal, ner_masking, punctuation_removal,
#                   lowercasing, whitespace_normalization 추가 ────
_ensure_data(); _ensure_models()

# spaCy 로딩 (NER masking 용)
try:
    nlp = spacy.load('en_core_web_sm', disable=['parser', 'lemmatizer'])
    NER_OK = True
except Exception as e:
    print(f"⚠ spaCy load failed: {e}. NER masking will be skipped.")
    NER_OK = False

# ────── 6가지 perturbation 함수 정의 ──────
def pert_typos(text, rate=0.05):
    if len(text) < 5: return text
    ws = text.split()
    out = []
    for w in ws:
        if random.random() < rate and len(w) > 2:
            i = random.randrange(len(w))
            if random.random() < 0.5:
                w = w[:i] + w[i+1:]            # delete
            else:
                w = w[:i] + random.choice(string.ascii_lowercase) + w[i:]  # insert
        out.append(w)
    return ' '.join(out)

def pert_source_marker_removal(text):
    # Reuters/AP/AFP/Bloomberg/CNN 등 흔한 source marker 제거
    pattern = (r'\b(?:\(?(?:Reuters|AP|AFP|Bloomberg|CNN|Forbes|NYTimes|'
               r'Newsday|USATODAY\.com|TechWeb)\)?[\s\-:,]*)')
    cleaned = re.sub(pattern, ' ', text, flags=re.IGNORECASE)
    return re.sub(r'\s{2,}', ' ', cleaned).strip()

def pert_punctuation_removal(text):
    return text.translate(str.maketrans('', '', string.punctuation))

def pert_lowercasing(text):
    return text.lower()

def pert_whitespace_normalization(text):
    return re.sub(r'\s+', ' ', text).strip()

def pert_ner_masking_single(text):
    if not NER_OK: return text
    doc = nlp(text)
    if not doc.ents: return text
    out, last = [], 0
    for ent in doc.ents:
        out.append(text[last:ent.start_char])
        out.append(f'[{ent.label_}]')
        last = ent.end_char
    out.append(text[last:])
    return ''.join(out)

def pert_ner_masking(texts):
    if not NER_OK: return texts
    out = []
    for doc, original in tqdm(zip(nlp.pipe(texts, batch_size=64), texts),
                                total=len(texts), desc='  NER', leave=False):
        if not doc.ents:
            out.append(original); continue
        s, last = [], 0
        for ent in doc.ents:
            s.append(original[last:ent.start_char])
            s.append(f'[{ent.label_}]')
            last = ent.end_char
        s.append(original[last:])
        out.append(''.join(s))
    return out


PERTURBATIONS_SINGLE = {
    'baseline':                 lambda t: t,
    'typo_insertion':           pert_typos,
    'source_marker_removal':    pert_source_marker_removal,
    'punctuation_removal':      pert_punctuation_removal,
    'lowercasing':              pert_lowercasing,
    'whitespace_normalization': pert_whitespace_normalization,
}

# Robustness 실험은 ORIGINAL(클렌징 전) 텍스트 사용 — 그래야 source marker 등이 살아 있음
te_labels = test_df[LABEL_COL].tolist()
te_texts_raw = test_df['title_desc_raw'].tolist()
b = trained_models[TEXT_COL]

# 평가 helper
def eval_all_models_on(texts, labels):
    out = {}
    preds = b['tfidf'].predict(texts)
    out['TF-IDF+LR'] = (accuracy_score(labels, preds),
                         f1_score(labels, preds, average='macro'))
    rr = predict_cnn(b['cnn'], texts, b['cnn_tok'], b['cnn_maxlen'])
    out['TextCNN'] = (accuracy_score(labels, rr['preds']),
                       f1_score(labels, rr['preds'], average='macro'))
    rr = predict_bert(b['bert'], texts, labels, get_probs=False)
    out['DistilBERT'] = (rr['acc'], rr['macro_f1'])
    return out

rob_results = {}
for pname, pfn in PERTURBATIONS_SINGLE.items():
    print(f"\n  Perturbation: {pname}")
    perturbed = [pfn(t) for t in te_texts_raw]
    rob_results[pname] = eval_all_models_on(perturbed, te_labels)
    for k, v in rob_results[pname].items():
        print(f"    {k:<12}: Acc={v[0]:.4f}  Macro-F1={v[1]:.4f}")

# NER masking (배치 처리)
if NER_OK:
    print("\n  Perturbation: ner_masking (running spaCy on test set, ~1-2 min)")
    perturbed_ner = pert_ner_masking(te_texts_raw)
    rob_results['ner_masking'] = eval_all_models_on(perturbed_ner, te_labels)
    for k, v in rob_results['ner_masking'].items():
        print(f"    {k:<12}: Acc={v[0]:.4f}  Macro-F1={v[1]:.4f}")
else:
    print("\n  ⚠ NER masking skipped (spaCy unavailable)")

# ────── Visualisation: Accuracy drop heatmap & bar chart ──────
model_names = ['TF-IDF+LR', 'TextCNN', 'DistilBERT']
all_perts = [k for k in rob_results.keys() if k != 'baseline']
drop_mat = np.zeros((len(all_perts), len(model_names)))
for i, pt in enumerate(all_perts):
    for j, mn in enumerate(model_names):
        drop_mat[i, j] = rob_results['baseline'][mn][0] - rob_results[pt][mn][0]

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(drop_mat, annot=True, fmt='.4f',
            xticklabels=model_names, yticklabels=all_perts,
            cmap='RdYlGn_r', center=0.01, vmin=0, vmax=0.10,
            linewidths=0.5, ax=ax)
ax.set_title(f'Accuracy Drop under Perturbations ({TEXT_COL}, test set)')
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(all_perts)); w = 0.25
for j, (mn, color) in enumerate(zip(model_names,
                                      ['#4878CF', '#6ACC65', '#D65F5F'])):
    ax.bar(x + j*w, drop_mat[:, j], w, label=mn, color=color,
           edgecolor='black', alpha=0.85)
ax.set_xticks(x + w); ax.set_xticklabels(all_perts, rotation=20, ha='right')
ax.set_ylabel('Accuracy Drop'); ax.set_title('Robustness — Accuracy Drop per Perturbation')
ax.legend(title='Model'); plt.tight_layout(); plt.show()

In [ ]:
# %% [Cell 9] Experiment 3 — Calibration + Qualitative HC-Wrong Predictions
# ──── 피드백 반영: high-confidence wrong prediction 정성 예시 분석 추가 ────
_ensure_data()
if 'trained_probs' not in dir() or len(trained_probs) < 3:
    trained_probs = {m: all_results[m][TEXT_COL]['probs']
                     for m in ['TF-IDF+LR', 'TextCNN', 'DistilBERT']}

def compute_ece(probs, labels, n_bins=10):
    confs = np.max(probs, axis=1)
    preds = np.argmax(probs, axis=1)
    acc   = (preds == np.asarray(labels)).astype(float)
    edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0; bins = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (confs > lo) & (confs <= hi)
        if m.sum() > 0:
            a = acc[m].mean(); c = confs[m].mean()
            ece += abs(a - c) * m.mean()
            bins.append((lo, hi, c, a))
    return ece, bins

true_labels_np = np.array(test_df[LABEL_COL].tolist())

# (a) Reliability diagrams
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (mname, probs) in zip(axes, trained_probs.items()):
    ece, bin_data = compute_ece(probs, true_labels_np)
    if bin_data:
        lo  = [b[0] for b in bin_data]
        w   = [b[1] - b[0] for b in bin_data]
        a_l = [b[3] for b in bin_data]
        c_l = [b[2] for b in bin_data]
        ax.bar(lo, a_l, width=w, align='edge', alpha=0.7, color='steelblue',
               label='Accuracy per bin')
        ax.plot(c_l, a_l, 'ro-', ms=5, label='Calibration curve')
    ax.plot([0,1],[0,1], 'k--', alpha=0.5, label='Perfect calibration')
    ax.set_xlim(0,1); ax.set_ylim(0,1)
    ax.set_xlabel('Confidence'); ax.set_ylabel('Accuracy')
    ax.set_title(f'{mname}\nECE = {ece:.4f}')
    ax.legend(fontsize=8)
plt.suptitle(f'Reliability Diagrams ({TEXT_COL}, test set)', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

ece_summary = {}
print('\nECE Summary:')
for mname, probs in trained_probs.items():
    ece, _ = compute_ece(probs, true_labels_np)
    ece_summary[mname] = ece
    print(f'  {mname:<12}: ECE = {ece:.4f}')

# (b) Confidence distribution (correct vs wrong)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (mname, probs) in zip(axes, trained_probs.items()):
    confs   = probs.max(axis=1)
    preds   = probs.argmax(axis=1)
    correct = (preds == true_labels_np)
    ax.hist(confs[correct],  bins=25, alpha=0.6, color='green',
            label='Correct', density=True)
    ax.hist(confs[~correct], bins=25, alpha=0.6, color='red',
            label='Wrong',   density=True)
    ax.axvline(0.90, color='black', linestyle='--', lw=1.5, label='Conf=0.90')
    ax.set_xlabel('Confidence'); ax.set_ylabel('Density')
    ax.set_title(f'{mname}'); ax.legend(fontsize=8)
plt.suptitle('Confidence Distribution: Correct vs Wrong', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

# (c) ── 피드백 반영: 정성적 High-Confidence Wrong Prediction 분석 ──
CONF_THR = 0.90
FAILURE_TAGS = ['ambiguous_topic', 'misleading_title', 'keyword_trap',
                'entity_shortcut', 'source_artifact', 'label_noise',
                'insufficient_context']

print(f"\n{'='*70}")
print(f"  High-Confidence (≥{CONF_THR}) Wrong Predictions — Qualitative Analysis")
print('='*70)

hc_wrong_examples = {}    # model → list of (idx, true, pred, conf, text)
for mname, probs in trained_probs.items():
    confs = probs.max(axis=1)
    preds = probs.argmax(axis=1)
    mask  = (preds != true_labels_np) & (confs >= CONF_THR)
    indices = np.where(mask)[0]
    hc_wrong_examples[mname] = []
    print(f"\n[{mname}]  HC-wrong: {len(indices):,} / {len(preds):,}  "
          f"({100*len(indices)/len(preds):.2f}%)")
    # 가장 confidence 가 높은 wrong 예시 상위 5개
    sort_order = indices[np.argsort(-confs[indices])][:5]
    for idx in sort_order:
        text_short = test_df['title_desc'].iloc[idx][:180]
        hc_wrong_examples[mname].append({
            'idx':   int(idx),
            'true':  CLASSES[true_labels_np[idx]],
            'pred':  CLASSES[preds[idx]],
            'conf':  float(confs[idx]),
            'text':  test_df['title_desc'].iloc[idx],
        })
        print(f"  idx={idx}  True={CLASSES[true_labels_np[idx]]:<10} "
              f"Pred={CLASSES[preds[idx]]:<10} Conf={confs[idx]:.3f}")
        print(f"    \"{text_short}\"")

# Failure category 분포 (자동 휴리스틱 태깅)
def auto_tag_failure(text, true_lbl, pred_lbl):
    tags = []
    if '(reuters)' in text.lower() or '(ap)' in text.lower():
        tags.append('source_artifact')
    if any(w in text.lower() for w in ['said', 'reported']):
        tags.append('insufficient_context')
    if len(text.split()) < 15:
        tags.append('insufficient_context')
    if true_lbl == 'Business' and pred_lbl == 'Sci/Tech':
        tags.append('ambiguous_topic')
    if true_lbl == 'Business' and pred_lbl == 'World':
        tags.append('ambiguous_topic')
    if any(c.isupper() for c in text[:10].replace(' ','')):
        tags.append('entity_shortcut')
    return tags or ['unclassified']

print(f"\n{'─'*70}\n  Failure-tag heuristic counts\n{'─'*70}")
for mname, examples in hc_wrong_examples.items():
    cnt = Counter()
    for ex in examples:
        for t in auto_tag_failure(ex['text'], ex['true'], ex['pred']):
            cnt[t] += 1
    print(f"  {mname:<12}: {dict(cnt)}")

In [ ]:
# %% [Cell 10] Experiment 4 — Masking Analysis (3 Techniques × 3 Models)
# ──── 피드백 반영: Random + Keyword + Attribution-based 모두 구현 ────
_ensure_data(); _ensure_models()
b = trained_models[TEXT_COL]
te_texts  = test_df[TEXT_COL].tolist()
te_labels = test_df[LABEL_COL].tolist()

# ───── (a) Class keyword 추출 (training data) ─────
kw_vect = TfidfVectorizer(ngram_range=(1,1), max_features=30_000, sublinear_tf=True)
X_kw    = kw_vect.fit_transform(train_df[TEXT_COL])
kw_feat = kw_vect.get_feature_names_out()
CLASS_KEYWORDS = {}
for c in range(NUM_CLASSES):
    mask = (train_df[LABEL_COL].values == c)
    mean_tfidf = X_kw[mask].mean(axis=0).A1
    top_idx = np.argsort(mean_tfidf)[-100:][::-1]
    CLASS_KEYWORDS[c] = set(kw_feat[top_idx])
ALL_KEYWORDS = set().union(*CLASS_KEYWORDS.values())
print(f"Unique class keywords: {len(ALL_KEYWORDS)}")

# ───── (b) Masking functions ─────
def mask_random(text, rate=0.30, seed=0):
    rng = random.Random(seed)
    ws = text.split()
    if not ws: return text
    n = max(1, int(len(ws) * rate))
    drop = set(rng.sample(range(len(ws)), min(n, len(ws))))
    return ' '.join(w for i, w in enumerate(ws) if i not in drop)

def mask_class_keywords(text, label):
    return ' '.join(w for w in text.split() if w.lower() not in CLASS_KEYWORDS[label])

def get_attr_tfidf(text, pipe, label):
    """TF-IDF + LR 의 단어별 |coef * tfidf|"""
    vocab = pipe.named_steps['tfidf'].vocabulary_
    tfv   = pipe.named_steps['tfidf'].transform([text])
    coef  = pipe.named_steps['clf'].coef_[label]
    scores = {}
    for w in set(text.split()):
        if w.lower() in vocab:
            idx = vocab[w.lower()]
            scores[w] = abs(coef[idx]) * tfv[0, idx]
        else:
            scores[w] = 0.0
    return scores

def mask_top_k_words(text, scores, k=5):
    if not scores: return text
    top = {w for w, _ in sorted(scores.items(),
                                  key=lambda x: x[1], reverse=True)[:k]}
    return ' '.join(w for w in text.split() if w not in top)

# ───── (c) 평가 헬퍼 ─────
def eval_tfidf_on(texts, labels):
    preds = b['tfidf'].predict(texts)
    return accuracy_score(labels, preds), f1_score(labels, preds, average='macro')

def eval_cnn_on(texts, labels):
    rr = predict_cnn(b['cnn'], texts, b['cnn_tok'], b['cnn_maxlen'])
    return accuracy_score(labels, rr['preds']), f1_score(labels, rr['preds'], average='macro')

def eval_bert_on(texts, labels):
    rr = predict_bert(b['bert'], texts, labels, get_probs=False)
    return rr['acc'], rr['macro_f1']

mask_results = {'baseline': {}, 'random': {}, 'keyword': {}, 'attribution': {}}

# ── Baseline ──
for mn in ['TF-IDF+LR', 'TextCNN', 'DistilBERT']:
    r = all_results[mn][TEXT_COL]
    mask_results['baseline'][mn] = (r['acc'], r['macro_f1'])

# ── 1. Random masking (3 runs averaged) ──
print("\n[1/3] Random Masking (30% words, 3 runs avg)")
for mn, fn in [('TF-IDF+LR', eval_tfidf_on),
                ('TextCNN',   eval_cnn_on),
                ('DistilBERT',eval_bert_on)]:
    accs, f1s = [], []
    for run in range(3):
        masked = [mask_random(t, rate=0.30, seed=SEED+run) for t in te_texts]
        a, f = fn(masked, te_labels)
        accs.append(a); f1s.append(f)
    mask_results['random'][mn] = (np.mean(accs), np.mean(f1s))
    print(f"  {mn:<12}: Acc={np.mean(accs):.4f}  Macro-F1={np.mean(f1s):.4f}")

# ── 2. Keyword masking ──
print("\n[2/3] Keyword Masking (class-indicative words removed)")
masked_kw = [mask_class_keywords(t, l) for t, l in zip(te_texts, test_df[LABEL_COL])]
for mn, fn in [('TF-IDF+LR', eval_tfidf_on),
                ('TextCNN',   eval_cnn_on),
                ('DistilBERT',eval_bert_on)]:
    a, f = fn(masked_kw, te_labels)
    mask_results['keyword'][mn] = (a, f)
    print(f"  {mn:<12}: Acc={a:.4f}  Macro-F1={f:.4f}")

# ── 3. Attribution-based masking ──
# 비용을 줄이기 위해 subset (300 samples) 사용. 모델 모두 동일 subset.
print("\n[3/3] Attribution-based Masking (leave-one-out, subset=300)")
ATTR_N = 300
random.seed(SEED)
attr_idx = random.sample(range(len(te_texts)), ATTR_N)
attr_texts  = [te_texts[i]  for i in attr_idx]
attr_labels = [te_labels[i] for i in attr_idx]

# TF-IDF: coefficient-based attribution (analytical, fast)
tfidf_preds = b['tfidf'].predict(attr_texts)
attr_masked_tfidf = [
    mask_top_k_words(t, get_attr_tfidf(t, b['tfidf'], int(pl)), k=5)
    for t, pl in zip(attr_texts, tfidf_preds)
]
a, f = eval_tfidf_on(attr_masked_tfidf, attr_labels)
mask_results['attribution']['TF-IDF+LR'] = (a, f)
print(f"  TF-IDF+LR  : Acc={a:.4f}  Macro-F1={f:.4f}")

# TextCNN: occlusion-based attribution
def occlusion_cnn(texts, labels, k=5):
    out_texts = []
    cnn_base = predict_cnn(b['cnn'], texts, b['cnn_tok'], b['cnn_maxlen'])
    base_probs = cnn_base['probs']
    base_preds = cnn_base['preds']
    for i, (text, pred) in enumerate(tqdm(zip(texts, base_preds),
                                            total=len(texts),
                                            desc='  CNN occlusion', leave=False)):
        words = text.split()
        if len(words) <= 1:
            out_texts.append(text); continue
        base_conf = base_probs[i, pred]
        variants = [' '.join(words[:j] + words[j+1:]) for j in range(len(words))]
        var_probs = predict_cnn(b['cnn'], variants,
                                 b['cnn_tok'], b['cnn_maxlen'])['probs']
        drops = base_conf - var_probs[:, pred]
        top_j = np.argsort(drops)[-k:]
        out_texts.append(' '.join(w for j, w in enumerate(words) if j not in top_j) or text)
    return out_texts

attr_masked_cnn = occlusion_cnn(attr_texts, attr_labels, k=5)
a, f = eval_cnn_on(attr_masked_cnn, attr_labels)
mask_results['attribution']['TextCNN'] = (a, f)
print(f"  TextCNN    : Acc={a:.4f}  Macro-F1={f:.4f}")

# DistilBERT: occlusion-based attribution
def occlusion_bert(texts, labels, k=5):
    out_texts = []
    base = predict_bert(b['bert'], texts, labels, get_probs=True)
    base_probs, base_preds = base['probs'], base['preds']
    for i, (text, pred) in enumerate(tqdm(zip(texts, base_preds),
                                            total=len(texts),
                                            desc='  BERT occlusion', leave=False)):
        words = text.split()
        if len(words) <= 1:
            out_texts.append(text); continue
        base_conf = base_probs[i, pred]
        variants = [' '.join(words[:j] + words[j+1:]) for j in range(len(words))]
        var_probs = predict_bert(b['bert'], variants,
                                  [0]*len(variants), get_probs=True)['probs']
        drops = base_conf - var_probs[:, pred]
        top_j = np.argsort(drops)[-k:]
        out_texts.append(' '.join(w for j, w in enumerate(words) if j not in top_j) or text)
    return out_texts

attr_masked_bert = occlusion_bert(attr_texts, attr_labels, k=5)
a, f = eval_bert_on(attr_masked_bert, attr_labels)
mask_results['attribution']['DistilBERT'] = (a, f)
print(f"  DistilBERT : Acc={a:.4f}  Macro-F1={f:.4f}")

# ───── Visualisation: 3 masking × 3 models grouped bar ─────
mask_types = ['baseline', 'random', 'keyword', 'attribution']
colors_m   = ['#4878CF', '#6ACC65', '#D65F5F', '#B47CC7']
model_names = ['TF-IDF+LR', 'TextCNN', 'DistilBERT']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, met_idx, met_name in zip(axes, [0, 1], ['Accuracy', 'Macro-F1']):
    x = np.arange(len(model_names)); w = 0.20
    for i, (mt, color) in enumerate(zip(mask_types, colors_m)):
        vals = [mask_results[mt].get(mn, (0, 0))[met_idx] for mn in model_names]
        ax.bar(x + i*w, vals, w, label=mt, color=color,
               edgecolor='black', alpha=0.85)
    ax.set_xticks(x + 1.5*w); ax.set_xticklabels(model_names)
    ax.set_ylabel(met_name); ax.set_title(f'{met_name} under Masking')
    ax.set_ylim(0.4, 1.0); ax.legend(title='Masking', fontsize=8)
plt.suptitle('Masking Experiments — 3 Techniques × 3 Models', fontsize=13)
plt.tight_layout(); plt.show()

# ────── Drop table ──────
print(f"\n{'='*70}\n  Accuracy Drop from Baseline (Masking)\n{'='*70}")
for mt in ['random', 'keyword', 'attribution']:
    print(f"\n  [{mt}]")
    for mn in model_names:
        base = mask_results['baseline'][mn][0]
        cur  = mask_results[mt][mn][0]
        print(f"    {mn:<12}: {base:.4f} → {cur:.4f}  (Δ={base-cur:+.4f})")

In [ ]:
# %% [Cell 11] Experiment 5 — Learning Curve Analysis  (NEW)
# ──── 피드백 반영: 서브샘플 학습으로 learning curve 작성 ────
# NOTE: 시간 절약을 위해 모드는 TEXT_COL(title_desc) 만 사용.
# 데이터 크기별 학습 후 결과 저장.
_ensure_data()

TRAIN_SIZES   = [1_000, 3_000, 5_000, 10_000, 30_000]   # full size 추가는 아래에서
CNN_LC_EPOCHS = 3
BERT_LC_EPOCHS= 2   # 시간 절약
BERT_LC_MAX   = 30_000   # 30K 초과 사이즈에 대해서는 학습 시간 부담으로 skip
LC_CACHE = os.path.join(LC_CKPT_DIR, 'learning_curve_results.pkl')

if os.path.exists(LC_CACHE):
    print(f"Loading cached learning-curve results from {LC_CACHE}")
    with open(LC_CACHE, 'rb') as f:
        lc_results = pickle.load(f)
else:
    lc_results = {m: {} for m in ['TF-IDF+LR', 'TextCNN', 'DistilBERT']}

for size in TRAIN_SIZES:
    print(f"\n--- training size = {size:,} ---")
    sub = train_df.sample(size, random_state=SEED).reset_index(drop=True)
    sub_texts  = sub[TEXT_COL].tolist()
    sub_labels = sub[LABEL_COL].tolist()
    te_texts   = test_df[TEXT_COL].tolist()
    te_labels  = test_df[LABEL_COL].tolist()

    # TF-IDF + LR
    if size not in lc_results['TF-IDF+LR']:
        pipe = train_tfidf_lr(sub_texts, sub_labels)
        preds = pipe.predict(te_texts)
        lc_results['TF-IDF+LR'][size] = (
            accuracy_score(te_labels, preds),
            f1_score(te_labels, preds, average='macro'))
        print(f"  TF-IDF+LR  : Acc={lc_results['TF-IDF+LR'][size][0]:.4f}")

    # TextCNN
    if size not in lc_results['TextCNN']:
        m_cnn, tok_cnn = train_cnn(sub_texts, sub_labels,
                                     val_df[TEXT_COL].tolist(),
                                     val_df[LABEL_COL].tolist(),
                                     max_len=CNN_MAX_LEN,
                                     epochs=CNN_LC_EPOCHS,
                                     verbose=False)
        rr = predict_cnn(m_cnn, te_texts, tok_cnn, CNN_MAX_LEN, get_probs=False)
        lc_results['TextCNN'][size] = (
            accuracy_score(te_labels, rr['preds']),
            f1_score(te_labels, rr['preds'], average='macro'))
        print(f"  TextCNN    : Acc={lc_results['TextCNN'][size][0]:.4f}")
        del m_cnn; torch.cuda.empty_cache()

    # DistilBERT (시간 부담으로 30K 이하만)
    if size <= BERT_LC_MAX and size not in lc_results['DistilBERT']:
        m_bert = train_bert(sub_texts, sub_labels,
                              val_df[TEXT_COL].tolist(),
                              val_df[LABEL_COL].tolist(),
                              epochs=BERT_LC_EPOCHS, verbose=False)
        rr = predict_bert(m_bert, te_texts, te_labels, get_probs=False)
        lc_results['DistilBERT'][size] = (rr['acc'], rr['macro_f1'])
        print(f"  DistilBERT : Acc={lc_results['DistilBERT'][size][0]:.4f}")
        del m_bert; torch.cuda.empty_cache()

    # save intermediate
    with open(LC_CACHE, 'wb') as f:
        pickle.dump(lc_results, f)

# full-size 결과 추가 (Cell 6 에서 평가된 본학습 결과 재사용)
FULL_SIZE = len(train_df)
lc_results['TF-IDF+LR'][FULL_SIZE] = (
    all_results['TF-IDF+LR'][TEXT_COL]['acc'],
    all_results['TF-IDF+LR'][TEXT_COL]['macro_f1'])
lc_results['TextCNN'][FULL_SIZE] = (
    all_results['TextCNN'][TEXT_COL]['acc'],
    all_results['TextCNN'][TEXT_COL]['macro_f1'])
lc_results['DistilBERT'][FULL_SIZE] = (
    all_results['DistilBERT'][TEXT_COL]['acc'],
    all_results['DistilBERT'][TEXT_COL]['macro_f1'])

with open(LC_CACHE, 'wb') as f:
    pickle.dump(lc_results, f)

# ────── Plot ──────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_cfg = {
    'TF-IDF+LR': ('#4878CF', 'o'),
    'TextCNN':   ('#6ACC65', 's'),
    'DistilBERT':('#D65F5F', '^'),
}
for ax, met_idx, met_name in zip(axes, [0, 1], ['Accuracy', 'Macro-F1']):
    for mname, (color, marker) in plot_cfg.items():
        sizes = sorted(lc_results[mname].keys())
        vals  = [lc_results[mname][s][met_idx] for s in sizes]
        ax.plot(sizes, vals, marker=marker, color=color,
                linewidth=2, markersize=7, label=mname)
    ax.set_xscale('log')
    ax.set_xlabel('Training Set Size (log)')
    ax.set_ylabel(met_name)
    ax.set_title(f'Learning Curve — {met_name}')
    ax.legend(); ax.grid(True, alpha=0.3)
plt.suptitle('Learning Curves (title_desc, test set)', fontsize=13)
plt.tight_layout(); plt.show()

print("\nLearning curve summary:")
for mn in ['TF-IDF+LR', 'TextCNN', 'DistilBERT']:
    print(f"\n[{mn}]")
    for s in sorted(lc_results[mn].keys()):
        print(f"  size={s:>7,}  Acc={lc_results[mn][s][0]:.4f}  "
              f"F1={lc_results[mn][s][1]:.4f}")

In [ ]:
# %% [Cell 12] TF-IDF + LR — Top Coefficient Features per Class  (NEW)
# ──── 피드백 반영: LR coefficient 기반 top feature 분석 ────
_ensure_models()
pipe_td = trained_models[TEXT_COL]['tfidf']
feat_names = pipe_td.named_steps['tfidf'].get_feature_names_out()
coefs      = pipe_td.named_steps['clf'].coef_   # (4, n_features)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
TOP_K = 20
for cls_idx, ax in enumerate(axes.flatten()):
    top_idx   = np.argsort(coefs[cls_idx])[-TOP_K:][::-1]
    top_words = [feat_names[i]      for i in top_idx]
    top_vals  = [coefs[cls_idx][i]  for i in top_idx]
    ax.barh(range(TOP_K), top_vals[::-1], color='steelblue', edgecolor='black')
    ax.set_yticks(range(TOP_K))
    ax.set_yticklabels(top_words[::-1], fontsize=9)
    ax.set_title(f'Top Features: {CLASSES[cls_idx]}', fontsize=11)
    ax.set_xlabel('Coefficient')
plt.suptitle('TF-IDF + LR — Top 20 Coefficients per Class', fontsize=14)
plt.tight_layout(); plt.show()

# 텍스트 출력
print("\n=== Top 10 features per class ===")
for cls_idx, cls in enumerate(CLASSES):
    top_idx = np.argsort(coefs[cls_idx])[-10:][::-1]
    words   = [f"{feat_names[i]}({coefs[cls_idx][i]:.2f})" for i in top_idx]
    print(f"  {cls:<10}: {', '.join(words)}")

In [ ]:
# %% [Cell 13] TextCNN — Filter Activation / Top Phrases  (NEW)
# ──── 피드백 반영: CNN filter activation 예시 ────
_ensure_data(); _ensure_models()
b = trained_models[TEXT_COL]
cnn_model, cnn_tok, cnn_maxlen = b['cnn'], b['cnn_tok'], b['cnn_maxlen']

# Test 데이터 subset 으로 filter activation 수집
SUBSET = 500
sub_texts = test_df[TEXT_COL].iloc[:SUBSET].tolist()

# Top filter activations 찾기: 각 conv layer 별로
def get_top_phrases_per_filter(model, texts, tokenizer, max_len,
                                  top_k_per_filter=3, top_filters=5):
    model.eval()
    ds = CNNDataset(texts, [0]*len(texts), tokenizer, max_len)
    loader = DataLoader(ds, batch_size=64, shuffle=False)
    # collect activations per filter
    # filter_results[layer_idx][filter_idx] = list of (activation, phrase)
    results = {fs: defaultdict(list) for fs in model.filter_sizes}
    all_inputs = []
    with torch.no_grad():
        for batch in loader:
            ids = batch['input_ids'].to(DEVICE)
            all_inputs.append(ids.cpu())
            _, activations = model(ids, return_activations=True)
            for layer_idx, fs in enumerate(model.filter_sizes):
                act = activations[layer_idx]   # (B, n_filters, L-fs+1)
                # 각 sample 별로 각 filter 의 max activation 위치
                for s_idx in range(act.size(0)):
                    text_words = texts[s_idx + len(all_inputs[-1])*0].split()[:max_len]
                    for f_idx in range(act.size(1)):
                        a = act[s_idx, f_idx]                # (L-fs+1,)
                        max_val, max_pos = a.max().item(), a.argmax().item()
                        # phrase 추출
                        phrase = ' '.join(text_words[max_pos:max_pos+fs])
                        results[fs][f_idx].append((max_val, phrase))
    # 정렬 + 상위 K
    output = {}
    for fs in model.filter_sizes:
        # filter 별 평균 max activation 으로 정렬, 상위 N filter 선택
        filter_scores = [(f_idx, np.mean([v for v, _ in vs[:50]]))
                          for f_idx, vs in results[fs].items()]
        filter_scores.sort(key=lambda x: -x[1])
        top_filters_list = []
        for f_idx, _ in filter_scores[:top_filters]:
            phrases = sorted(results[fs][f_idx], key=lambda x: -x[0])[:top_k_per_filter]
            top_filters_list.append({'filter': f_idx, 'phrases': phrases})
        output[fs] = top_filters_list
    return output

print(f"Collecting filter activations on {SUBSET} samples ...")
filter_phrases = get_top_phrases_per_filter(cnn_model, sub_texts, cnn_tok, cnn_maxlen,
                                              top_k_per_filter=3, top_filters=5)

print(f"\n=== TextCNN Filter Activation Analysis ===")
for fs, filters_list in filter_phrases.items():
    print(f"\n[Filter size = {fs}]   (most activated 5 filters)")
    for entry in filters_list:
        print(f"  Filter #{entry['filter']:>3}: "
              + ', '.join([f'"{p}"({v:.2f})' for v, p in entry['phrases']]))

In [ ]:
# %% [Cell 14] DistilBERT — Attention Analysis  (NEW)
# ──── 피드백 반영: attention / token attribution 분석 ────
_ensure_data(); _ensure_models()

# Attention 출력 가능한 모델 새로 구성 (체크포인트 재로드)
print("Loading DistilBERT with attention output enabled ...")
attn_model = build_bert_model(output_attentions=True)
attn_model.load_state_dict(trained_models[TEXT_COL]['bert'].state_dict())
attn_model.eval()

# Attention 분석할 예시 선택: high-confidence wrong + 정답 예시 섞어서
ANALYSIS_N = 5
np.random.seed(SEED)
# wrong-prediction examples (높은 confidence)
bert_probs = trained_probs['DistilBERT']
bert_preds = np.argmax(bert_probs, axis=1)
true_np    = np.array(test_df[LABEL_COL].tolist())
confs      = bert_probs.max(axis=1)
hc_wrong_idx = np.where((bert_preds != true_np) & (confs >= 0.85))[0]
if len(hc_wrong_idx) >= ANALYSIS_N:
    sel_idx = np.random.choice(hc_wrong_idx, ANALYSIS_N, replace=False)
else:
    sel_idx = np.random.choice(len(test_df), ANALYSIS_N, replace=False)

def get_token_attention(model, text, target_layer=-1):
    """마지막 layer 의 [CLS] → 다른 토큰들 attention 평균(across heads)."""
    enc = bert_tokenizer(text, max_length=BERT_MAX_LEN, truncation=True,
                          padding='max_length', return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = model(**enc, output_attentions=True)
    # out.attentions: tuple of (B, n_heads, L, L), one per layer
    last_attn = out.attentions[target_layer][0]   # (n_heads, L, L)
    cls_attn  = last_attn[:, 0, :].mean(dim=0)    # (L,) heads-averaged
    tokens = bert_tokenizer.convert_ids_to_tokens(enc['input_ids'][0])
    return tokens, cls_attn.cpu().numpy(), out.logits[0].cpu().numpy()

# Visualisation: 각 example 별 token-level attention heatmap
fig, axes = plt.subplots(ANALYSIS_N, 1, figsize=(16, 2.0 * ANALYSIS_N))
if ANALYSIS_N == 1: axes = [axes]
for ax, idx in zip(axes, sel_idx):
    text = test_df['title_desc'].iloc[int(idx)][:300]
    tokens, attn, logits = get_token_attention(attn_model, text)
    # PAD 토큰 제거
    valid_mask = np.array([t != bert_tokenizer.pad_token for t in tokens])
    tokens_v = [t for t, m in zip(tokens, valid_mask) if m]
    attn_v   = attn[valid_mask]
    # 최대 40 토큰만 시각화
    if len(tokens_v) > 40:
        tokens_v = tokens_v[:40]; attn_v = attn_v[:40]
    pred = int(np.argmax(logits)); true = int(test_df[LABEL_COL].iloc[int(idx)])

    sns.heatmap(attn_v[np.newaxis, :], xticklabels=tokens_v,
                yticklabels=['[CLS] attn'], cmap='YlOrRd', cbar=True,
                ax=ax, vmin=0, vmax=attn_v.max())
    ax.set_xticklabels(tokens_v, rotation=45, ha='right', fontsize=8)
    ax.set_title(f'idx={idx}  True={CLASSES[true]}  Pred={CLASSES[pred]}',
                 fontsize=10)
plt.suptitle('DistilBERT — [CLS] Token Attention (last layer, heads-averaged)',
             fontsize=13, y=1.005)
plt.tight_layout(); plt.show()

# 텍스트로도 출력
print(f"\n=== Top-5 tokens attended to by [CLS] ===")
for idx in sel_idx:
    text = test_df['title_desc'].iloc[int(idx)][:200]
    tokens, attn, logits = get_token_attention(attn_model, text)
    valid_mask = np.array([t != bert_tokenizer.pad_token for t in tokens])
    tokens_v = [t for t, m in zip(tokens, valid_mask) if m]
    attn_v   = attn[valid_mask]
    # CLS, SEP 제외
    top5 = np.argsort(-attn_v)[1:6]
    pred = int(np.argmax(logits)); true = int(test_df[LABEL_COL].iloc[int(idx)])
    print(f"\n  idx={idx}  True={CLASSES[true]}  Pred={CLASSES[pred]}")
    print(f"    text: \"{text[:120]}\"")
    print(f"    top tokens: " +
          ', '.join([f"{tokens_v[i]}({attn_v[i]:.3f})" for i in top5]))

del attn_model; torch.cuda.empty_cache()

In [ ]:
# %% [Cell 15] Ambiguous / Mixed-Signal Example Analysis  (NEW)
# ──── 피드백 반영: ambiguous / mixed signal 예시 분석 ────
# Strategy: 세 모델의 예측이 서로 다른 sample 을 ambiguous case 로 간주
_ensure_data()
tfidf_preds = np.argmax(all_results['TF-IDF+LR'][TEXT_COL]['probs'], axis=1)
cnn_preds   = np.argmax(all_results['TextCNN'  ][TEXT_COL]['probs'], axis=1)
bert_preds  = np.argmax(all_results['DistilBERT'][TEXT_COL]['probs'], axis=1)
true_np     = np.array(test_df[LABEL_COL].tolist())

# (1) 세 모델이 모두 다른 예측을 한 경우
all_diff = np.where((tfidf_preds != cnn_preds) & (cnn_preds != bert_preds)
                    & (tfidf_preds != bert_preds))[0]

# (2) 두 모델 vs 한 모델 — Business/Sci/Tech, Business/World 흔한 헷갈림
biz_sci_mix = np.where(((true_np == 2) & ((tfidf_preds == 3) | (cnn_preds == 3) | (bert_preds == 3)))
                       | ((true_np == 3) & ((tfidf_preds == 2) | (cnn_preds == 2) | (bert_preds == 2))))[0]

print("=" * 70)
print("  Ambiguous / Mixed-Signal Examples")
print("=" * 70)

print(f"\n[1] All-three-models-disagree examples (n={len(all_diff)}):")
for idx in all_diff[:5]:
    print(f"\n  idx={idx}  True={CLASSES[true_np[idx]]}")
    print(f"    TF-IDF→{CLASSES[tfidf_preds[idx]]}  "
          f"CNN→{CLASSES[cnn_preds[idx]]}  "
          f"BERT→{CLASSES[bert_preds[idx]]}")
    print(f"    \"{test_df['title_desc'].iloc[int(idx)][:180]}\"")

print(f"\n[2] Business ↔ Sci/Tech confusion examples (n={len(biz_sci_mix)}):")
for idx in biz_sci_mix[:5]:
    print(f"\n  idx={idx}  True={CLASSES[true_np[idx]]}")
    print(f"    TF-IDF→{CLASSES[tfidf_preds[idx]]}  "
          f"CNN→{CLASSES[cnn_preds[idx]]}  "
          f"BERT→{CLASSES[bert_preds[idx]]}")
    print(f"    \"{test_df['title_desc'].iloc[int(idx)][:180]}\"")

# Inter-model agreement matrix
print(f"\n[3] Inter-model Agreement (test set):")
print(f"  TF-IDF & TextCNN agree   : {(tfidf_preds == cnn_preds).mean()*100:.2f}%")
print(f"  TF-IDF & DistilBERT agree: {(tfidf_preds == bert_preds).mean()*100:.2f}%")
print(f"  TextCNN & DistilBERT agree: {(cnn_preds == bert_preds).mean()*100:.2f}%")
print(f"  All-three agree          : {((tfidf_preds == cnn_preds)&(cnn_preds == bert_preds)).mean()*100:.2f}%")

In [ ]:
# %% [Cell 16] FINAL SUMMARY DASHBOARD
print('=' * 75)
print('  FINAL SUMMARY — v26_05_19 (Feedback-Reflected)')
print('=' * 75)

# [1] Best input mode
if 'all_results' in dir():
    print('\n[1] Best Input Mode per Model:')
    print(f"  {'Model':<12}  {'Best Input':<20}  {'Acc':>8}  {'Macro-F1':>10}")
    for mname in ['TF-IDF+LR', 'TextCNN', 'DistilBERT']:
        best = max(all_results[mname], key=lambda m: all_results[mname][m]['macro_f1'])
        r = all_results[mname][best]
        print(f"  {mname:<12}  {best:<20}  {r['acc']:>8.4f}  {r['macro_f1']:>10.4f}")

    print(f'\n[2] {TEXT_COL} — Per-class F1:')
    h = f"  {'Model':<12}  {'Acc':>8}  {'Macro-F1':>10}"
    for cls in CLASSES: h += f"  {cls[:5]:>7}"
    print(h)
    for mname in ['TF-IDF+LR', 'TextCNN', 'DistilBERT']:
        r = all_results[mname][TEXT_COL]
        row = f"  {mname:<12}  {r['acc']:>8.4f}  {r['macro_f1']:>10.4f}"
        for v in r['per_f1']: row += f"  {v:>7.4f}"
        print(row)

# [3] Robustness
if 'rob_results' in dir():
    print('\n[3] Robustness — Avg Accuracy Drop per Perturbation:')
    model_names = ['TF-IDF+LR', 'TextCNN', 'DistilBERT']
    all_perts_local = [p for p in rob_results if p != 'baseline']
    for pt in all_perts_local:
        drops = [rob_results['baseline'][mn][0] - rob_results[pt][mn][0]
                 for mn in model_names]
        print(f"  {pt:<28}: avg_drop = {np.mean(drops):+.4f}")

# [4] Masking — 3 techniques
if 'mask_results' in dir():
    print('\n[4] Masking — Accuracy Drop per Technique:')
    print(f"  {'Model':<12}  {'Random':>10}  {'Keyword':>10}  {'Attribution':>13}")
    for mn in ['TF-IDF+LR', 'TextCNN', 'DistilBERT']:
        base = mask_results['baseline'][mn][0]
        rnd  = base - mask_results['random'    ][mn][0]
        kw   = base - mask_results['keyword'   ][mn][0]
        at   = base - mask_results['attribution'][mn][0]
        print(f"  {mn:<12}  {rnd:>+10.4f}  {kw:>+10.4f}  {at:>+13.4f}")

# [5] Calibration
if 'ece_summary' in dir():
    print('\n[5] Expected Calibration Error (ECE):')
    for mn, v in ece_summary.items():
        print(f'  {mn:<12}: ECE = {v:.4f}')

# [6] Learning curve
if 'lc_results' in dir():
    print('\n[6] Learning Curve (Accuracy at key sizes):')
    sizes_show = [1_000, 10_000, 30_000, len(train_df)]
    print(f"  {'Model':<12}", end='')
    for s in sizes_show: print(f"  {s:>8,}", end='')
    print()
    for mn in ['TF-IDF+LR', 'TextCNN', 'DistilBERT']:
        print(f"  {mn:<12}", end='')
        for s in sizes_show:
            v = lc_results[mn].get(s, (None,))[0]
            print(f"  {v:>8.4f}" if v is not None else f"  {'─':>8}", end='')
        print()

print('\n' + '=' * 75)
print('  All experiments complete — proposal fully covered.')
print('=' * 75)